# FabricDaxLoadTest — Queries

Editor for the **shared** DAX query catalog at
`LoadTests.Lakehouse/Files/queries.json`. This file is the *fallback*
that `LoadTest - <name>` notebooks use when no `queries.json` is
attached to their own *Resources* panel.

> **Per-test corpora belong in notebook resources, not here.** The
> canonical workflow is: in your `LoadTest - <name>` copy, drag the
> per-test `queries.json` onto the Resources panel — it travels with
> the saved notebook and the run is reproducible without depending on
> the shared catalog.
>
> Use this notebook only to edit the shared default that newly-created
> `LoadTest - <name>` copies pick up before they're customised.

The catalog file is either:

- a JSON list of strings: `["EVALUATE ROW(\"x\", 1)", ...]`, or
- a JSON list of objects with a `query` key:
  `[{"query": "EVALUATE ...", "name": "...", "tags": [...]}]` — extra
  fields are preserved on round-trip but ignored by the runner.

In [ ]:
# ── 1. Configuration ──────────────────────────────────────────────────────────
# Path to the query corpus, relative to `LoadTests.Lakehouse/Files/`. Use
# `queries.json` (default) for the shared catalog, or a per-test path like
# `tests/diad-baseline/queries.json`. An absolute `abfss://…` URL is also
# accepted as an escape hatch.
QUERIES_FILE = "queries.json"

import json, notebookutils
ctx = notebookutils.runtime.context
WS_ID = ctx["currentWorkspaceId"]
LH_ABFSS = f"abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com/LoadTests.Lakehouse"
QPATH = QUERIES_FILE if QUERIES_FILE.startswith("abfss://") else f"{LH_ABFSS}/Files/{QUERIES_FILE.lstrip('/')}"
print(f"Lakehouse: {LH_ABFSS}")
print(f"Catalog  : {QPATH}")

In [ ]:
# ── 2. Read the current catalog ───────────────────────────────────────────────
try:
    raw = notebookutils.fs.head(QPATH, 1024 * 1024 * 4)
    queries = json.loads(raw)
    print(f"Loaded {len(queries)} queries from {QPATH}")
except Exception as e:
    print(f"(no existing catalog — will create on save: {e})")
    queries = []

import pandas as pd
def to_df(qs):
    rows = []
    for i, q in enumerate(qs):
        if isinstance(q, str):
            rows.append({"i": i, "name": "", "tags": "", "query": q})
        else:
            rows.append({
                "i":     i,
                "name":  q.get("name", ""),
                "tags":  ",".join(q.get("tags", [])) if isinstance(q.get("tags"), list) else (q.get("tags") or ""),
                "query": q.get("query", ""),
            })
    return pd.DataFrame(rows)

df = to_df(queries)
display(df.head(50))

In [ ]:
# ── 3. Edit the catalog ───────────────────────────────────────────────────────
# Edit the Python literal below and re-run cells 3 + 4. Each entry is a string
# (just the DAX) or a dict with optional name/tags. Strings work for simple
# cases; switch to dicts when you want labels in the report.
new_queries = [
    # Replace these examples with your real corpus.
    "EVALUATE ROW(\"x\", 1)",
    {"name": "topn-sales", "tags": ["sales", "topn"],
     "query": "EVALUATE TOPN(10, SUMMARIZECOLUMNS('Date'[Year], \"Sales\", [Sales Amount]))"},
]
print(f"Prepared {len(new_queries)} queries.")
display(to_df(new_queries))

In [ ]:
# ── 4. Save back to OneLake ───────────────────────────────────────────────────
notebookutils.fs.put(QPATH, json.dumps(new_queries, indent=2), overwrite=True)
print(f"Saved {len(new_queries)} queries to {QPATH}")